# 05 Docking And Pose Filtering

Infer the binding grid from PDB 5AEP, prepare top candidates as 3D SDF, then create a Vina manifest once PDBQT files exist.


In [ ]:
# Install cell
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/EMD_V5_2_Hybrid"
REQ = f"{BASE}/requirements_v5_2_hybrid_colab.txt"
!pip -q install -r "{REQ}"


In [ ]:
# Config cell
import os
import sys
from pathlib import Path

BASE_PATH = Path(BASE)
sys.path.insert(0, str(BASE_PATH / "src"))

from emd_v5_2_hybrid.registry import ensure_project_tree, register_run, save_progress

ensure_project_tree(BASE_PATH)
print(f"Project base: {BASE_PATH}")


In [ ]:
# Tiny Debug mode
TINY_DEBUG = True
TINY_SIZE = 25
RANDOM_SEED = 42
print({"tiny_debug": TINY_DEBUG, "tiny_size": TINY_SIZE, "seed": RANDOM_SEED})


In [ ]:
# Resume/progress cell
NOTEBOOK_SLUG = "05_docking_and_pose_filtering"
PROGRESS_PATH = BASE_PATH / "00_project_registry" / f"progress_{NOTEBOOK_SLUG}.json"
save_progress(PROGRESS_PATH, {"last_completed_index": -1, "status": "started"})
print(f"Progress file: {PROGRESS_PATH}")


In [ ]:
# Step 1: infer grid from the co-crystallized QUP ligand and prepare candidate SDF files.
prep_script = BASE_PATH / "scripts" / "06_prepare_docking_inputs.py"
!python "{prep_script}" --base "{BASE}" --top-n 150

# Step 2: prepare receptor/ligand PDBQT files and build a Vina manifest.
pdbqt_script = BASE_PATH / "scripts" / "06_prepare_pdbqt.py"
manifest_script = BASE_PATH / "scripts" / "06_make_vina_manifest.py"
!python "{pdbqt_script}" --base "{BASE}"
!python "{manifest_script}" --base "{BASE}"

# Step 3: run Vina where the vina executable is available, then parse logs.
# run_script = BASE_PATH / "scripts" / "06_run_vina_manifest.py"
# parse_script = BASE_PATH / "scripts" / "06_parse_vina_results.py"
# !python "{run_script}" --base "{BASE}"
# !python "{parse_script}" --base "{BASE}"


In [ ]:
grid = BASE_PATH / "06_docking" / "receptor" / "docking_grid_5AEP_QUP.json"
sdf = BASE_PATH / "06_docking" / "ligands_sdf" / "candidates_for_docking.sdf"
receptor = BASE_PATH / "06_docking" / "receptor" / "jak2_prepared.pdbqt"
manifest = BASE_PATH / "06_docking" / "scores" / "vina_command_manifest.csv"
assert grid.exists() and grid.stat().st_size > 0
assert sdf.exists() and sdf.stat().st_size > 0
assert receptor.exists() and receptor.stat().st_size > 0
assert manifest.exists() and manifest.stat().st_size > 0
print(grid.read_text())
print("Docking prep validation passed. Vina scores remain the next gate.")


In [ ]:
print("Summary: docking grid and ligand SDF files are ready. Vina scores remain the next gate.")
